In [1]:
import json
import time
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
MODEL = "gpt-4o-mini"

print("환경 설정 완료!")


환경 설정 완료!


In [2]:
# messages = []

# messages.append()
llm.invoke('내 이름은 abc입니다.').content

'안녕하세요, abc님! 어떻게 도와드릴까요?'

In [3]:
llm.invoke('내 이름은 뭔가요?.').content

'죄송하지만, 당신의 이름은 알 수 없습니다. 개인 정보를 제공하지 않으시면 이름을 알 수 없습니다. 하지만 다른 질문이나 대화할 주제가 있다면 기꺼이 도와드리겠습니다!'

In [ ]:
# messages.append
# 1번턴 ~~~~ 100번턴   -> 최근 4개턴만 보자
# 1번턴 ~~ 10번턴 -> 요약 , 11번턴 ~20번턴 -> 요약

In [4]:
messages = [
    SystemMessage(content = '당신은 친절한 AI 어시스턴트입니다'),
    HumanMessage(content = '내 이름은 abc입니다. 반가워요')
]

response1 = llm.invoke(messages)
print(response1.content)

반가워요, abc님! 어떻게 도와드릴까요?


In [5]:
messages.append(AIMessage(content = response1.content))
messages.append(HumanMessage(content = '내 이름은 뭔가요?'))
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 이름은 abc입니다. 반가워요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반가워요, abc님! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름은 뭔가요?', additional_kwargs={}, response_metadata={})]

In [6]:
response2 = llm.invoke(messages)
print(response2.content)

당신의 이름은 abc입니다! 다른 질문이 있으신가요?


In [8]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

def count_tokens(messages):
    total = 0
    for msg in messages:
        total += len(enc.encode(msg.content))
        total += 4
    return total

conversation = [SystemMessage(content = '당신은 친절한 AI 어시스턴트입니다')]
sample_exchanges = [
    ('내 이름은 abc입니다. 반가워요', '반가워요, abc님! 어떻게 도와드릴까요?'),
    ('내 이름은 뭔가요?', '당신의 이름은 abc입니다! 다른 질문이 있으신가요?')
]

for i, (user_msg, ai_msg) in enumerate(sample_exchanges):
    conversation.append(HumanMessage(user_msg))
    conversation.append(AIMessage(ai_msg))
    tokens = count_tokens(conversation)
    
    print(f"{i} | {tokens} |")

0 | 50 |
1 | 82 |


In [ ]:
# Window : 대화 window -> 최신 맥락만 유지
# summarize : 10, 100 turn -> 요약 -> 새로 대화를 시작 : 디테일 유지 + 약간의 추가 토큰비용

In [9]:
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory

In [ ]:
# 내부적으로 memory 가지고 있다가 사용 / summarize : 
    
# LangGraph : 모든 knowledge / 파이프 : 하나하나의 state -> 개발자가 state  

In [10]:
memory = ConversationBufferMemory(return_messages=True)

/tmp/ipykernel_2868257/3614923015.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


In [11]:
memory.save_context(
    {'input' : '안녕하세요, 저는 abc입니다'},
    {'output' : '안녕하세요, abc님! 만나서 반갑습니다'}
)
memory.save_context(
    {'input' : '오늘 날씨가 좋네요!'},
    {'output' : '네, 정말 화창한 날씨입니다'}
)

In [13]:
history = memory.load_memory_variables({})

In [14]:
history

{'history': [HumanMessage(content='안녕하세요, 저는 abc입니다', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='네, 정말 화창한 날씨입니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [15]:
for msg in history['history']:
    print(msg.content)

안녕하세요, 저는 abc입니다
안녕하세요, abc님! 만나서 반갑습니다
오늘 날씨가 좋네요!
네, 정말 화창한 날씨입니다


In [16]:
memory = ConversationBufferMemory(return_messages=False)
memory.save_context(
    {'input' : '안녕하세요, 저는 abc입니다'},
    {'output' : '안녕하세요, abc님! 만나서 반갑습니다'}
)
memory.save_context(
    {'input' : '오늘 날씨가 좋네요!'},
    {'output' : '네, 정말 화창한 날씨입니다'}
)
history = memory.load_memory_variables({})
history

{'history': 'Human: 안녕하세요, 저는 abc입니다\nAI: 안녕하세요, abc님! 만나서 반갑습니다\nHuman: 오늘 날씨가 좋네요!\nAI: 네, 정말 화창한 날씨입니다'}

In [17]:
from langchain_core.chat_history import InMemoryChatMessageHistory

In [18]:
chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message('안녕하세요, 저는 abc입니다')
chat_history.add_ai_message('안녕하세요, abc님! 만나서 반갑습니다')
chat_history.add_user_message('오늘 날씨가 좋네요')
chat_history.add_ai_message('네, 정말 화창한 날씨입니다')

In [19]:
chat_history

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕하세요, 저는 abc입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='오늘 날씨가 좋네요', additional_kwargs={}, response_metadata={}), AIMessage(content='네, 정말 화창한 날씨입니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [20]:
for msg in chat_history.messages:
    print(msg.type, msg.content)

human 안녕하세요, 저는 abc입니다
ai 안녕하세요, abc님! 만나서 반갑습니다
human 오늘 날씨가 좋네요
ai 네, 정말 화창한 날씨입니다


In [21]:
chat_history.messages

[HumanMessage(content='안녕하세요, 저는 abc입니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='오늘 날씨가 좋네요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네, 정말 화창한 날씨입니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [22]:
chat_history.messages.append(HumanMessage(content = '내 이름이 뭔가요?'))
chat_history.messages

[HumanMessage(content='안녕하세요, 저는 abc입니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='오늘 날씨가 좋네요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네, 정말 화창한 날씨입니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭔가요?', additional_kwargs={}, response_metadata={})]

In [23]:
llm.invoke(chat_history.messages)

AIMessage(content='당신의 이름은 "abc"입니다. 다른 이름이나 별명이 있다면 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 63, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DNzmCn8BQi6g5HtcCKU9DtYY6vAdo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f14-c45e-7120-a710-db44b53b6276-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 21, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
messages = [
    *chat_history.messages,
    HumanMessage(content = '내 이름이 뭔가요?')
]
llm.invoke(messages)

AIMessage(content='당신의 이름은 "abc"입니다. 다른 질문이나 이야기하고 싶은 내용이 있으면 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 75, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DNzo6W3bPZa9ycJHpxeRfyqwG9Yd9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f16-9125-7890-b1da-06836e4c3472-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 24, 'total_tokens': 99, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
# Runnable
# RunnablePassthrough, RunnableLambda, RunnableParallel 
# RunnableWithMessageHistory

In [27]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


In [28]:
# llm = ChatOpenAI
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 AI 어시스턴트입니다'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])
chain = prompt | llm

In [29]:
store = {}
# store = {'session_id1' : xxxxxxxx, 'session_id2':  xxxxxx, 'session_id3' : InMemoryChatMessageHistory()}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key = 'input',
        history_messages_key = 'history'
)

In [30]:
config = {'configurable': {'session_id': 'user_001'}}
r1 = chain_with_history.invoke({'input' : '안녕하세요, 제 이름은 abc입니다'}, config=config)

In [31]:
r1

AIMessage(content='안녕하세요, abc님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 32, 'total_tokens': 45, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0DD9IPeb0WDdPj3vxCT6LFRYyBT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2e-5678-7e81-afd1-a6d0c3e578ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 13, 'total_tokens': 45, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [32]:
r2 = chain_with_history.invoke({'input' : '내 이름이 뭐라고 했죠?'}, config=config)

In [33]:
r2

AIMessage(content='당신의 이름은 abc입니다. 맞나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 61, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0EUsuf9CuhN04hhoxdzZUY87MUR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2f-8939-7122-8dca-43679670a8f4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 11, 'total_tokens': 72, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [34]:
store

{'user_001': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕하세요, 제 이름은 abc입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 32, 'total_tokens': 45, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0DD9IPeb0WDdPj3vxCT6LFRYyBT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2e-5678-7e81-afd1-a6d0c3e578ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 13, 'total_tokens': 45, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details':

In [35]:
r3 = chain_with_history.invoke({'input' : '오늘 뭐하면 좋을까요?'}, config=config)

In [37]:
r3.content

'오늘 할 수 있는 다양한 아이디어가 있습니다! 몇 가지 제안드릴게요:\n\n1. **산책이나 하이킹**: 자연을 즐기면서 운동도 할 수 있어요.\n2. **책 읽기**: 읽고 싶었던 책이나 새로운 주제의 책을 읽어보세요.\n3. **요리**: 새로운 레시피에 도전해 보거나 간단한 요리를 해보세요.\n4. **영화나 드라마 감상**: 보고 싶었던 영화를 보거나 재미있는 드라마 시리즈를 시작해 보세요.\n5. **취미 활동**: 그림 그리기, 음악 연주, 공예 등 취미에 시간을 투자해보세요.\n6. **친구나 가족과의 소통**: 전화나 영상통화로 소중한 사람들과 이야기 나누세요.\n\n어떤 활동이 마음에 드시나요?'

In [38]:
# llm = ChatOpenAI
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 전자제품 판매원입니다'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])
chain = prompt | llm

store = {}
# store = {'session_id1' : xxxxxxxx, 'session_id2':  xxxxxx, 'session_id3' : InMemoryChatMessageHistory()}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key = 'input',
        history_messages_key = 'history'
)

config = {'configurable': {'session_id': 'user_001'}}
r1 = chain_with_history.invoke({'input' : '안녕하세요, 노트북을 사려는데 어떤게 있나요?'}, config=config)

In [39]:
r1.content

'안녕하세요! 노트북을 구매하시려는군요. 용도에 따라 다양한 모델이 있습니다. 사용 목적에 맞춰 몇 가지 추천해드리겠습니다.\n\n1. **일반적인 사용 (웹 서핑, 문서 작업 등)**:\n   - **LG 그램**: 가벼운 무게와 긴 배터리 수명이 장점입니다. 이동성이 중요하다면 좋습니다.\n   - **삼성 노트북 9**: 슬림한 디자인과 성능이 뛰어나며, 화면이 선명합니다.\n\n2. **게이밍**:\n   - **레노버 리전 5**: 강력한 그래픽 카드와 프로세서로 게임에 최적화되어 있습니다.\n   - **Razer Blade 15**: 고급스러운 디자인과 뛰어난 성능을 자랑합니다.\n\n3. **작업용 (그래픽 디자인, 영상 편집 등)**:\n   - **애플 맥북 프로**: 뛰어난 디스플레이와 성능, 최적화된 소프트웨어 환경이 특징입니다.\n   - **MSI 크리에이터 시리즈**: 고성능 프로세서와 그래픽 카드로 크리에이티브 작업에 적합합니다.\n\n4. **가성비**:\n   - **아수스 비보북**: 좋은 성능에 가격이 합리적입니다. 일상적인 작업에 적합합니다.\n   - **HP 파빌리온**: 다양한 옵션이 있으며, 성능과 가격 밸런스가 좋습니다.\n\n어떤 용도로 사용하실지 말씀해주시면, 더 구체적인 추천을 해드릴 수 있습니다!'

In [40]:
r2 = chain_with_history.invoke({'input' : '가장 인기 있는 제품의 가격이 어떻게 되나요?'}, config=config)

In [41]:
r2.content

'인기 있는 노트북 제품들의 가격은 모델과 스펙에 따라 조금씩 다르지만, 대략적인 가격대는 다음과 같습니다:\n\n1. **LG 그램**:\n   - 가격 범위: 약 1,200,000원 ~ 2,000,000원\n   - 모델에 따라 다르지만, 가벼움과 긴 배터리 수명이 큰 장점입니다.\n\n2. **삼성 노트북 9**:\n   - 가격 범위: 약 1,000,000원 ~ 1,800,000원\n   - 디자인과 성능이 뛰어난 제품으로, 다양한 옵션이 있습니다.\n\n3. **레노버 리전 5** (게이밍):\n   - 가격 범위: 약 1,500,000원 ~ 2,500,000원\n   - 성능이 우수하며, 다양한 게임에 최적화되어 있습니다.\n\n4. **Razer Blade 15** (게이밍):\n   - 가격 범위: 약 2,000,000원 ~ 3,500,000원\n   - 고급스러운 디자인과 좋은 성능을 가지고 있습니다.\n\n5. **애플 맥북 프로**:\n   - 가격 범위: 약 1,900,000원 ~ 3,500,000원\n   - 고성능과 뛰어난 디자인, macOS 환경을 원하신다면 추천드립니다.\n\n6. **MSI 크리에이터 시리즈**:\n   - 가격 범위: 약 1,500,000원 ~ 2,800,000원\n   - 크리에이티브 작업에 적합한 성능을 제공합니다.\n\n가격은 판매처나 세일에 따라 달라질 수 있으니, 구매하시기 전에 여러 사이트를 비교해보시는 것이 좋습니다. 구매하시고자 하는 모델이나 더 구체적인 정보가 필요하시면 말씀해 주세요!'

In [42]:
r3 = chain_with_history.invoke({'input' : '좋아요, 2번째 제품으로 구입하겠습니다.'}, config=config)
r3.content

'삼성 노트북 9을 선택해 주셨군요! 정말 좋은 선택입니다. 삼성 노트북 9은 뛰어난 디자인, 성능, 그리고 경량화된 특성이 매우 매력적입니다.\n\n구매를 위해 고려하실 사항은 다음과 같습니다:\n\n1. **모델 선택**: 삼성 노트북 9은 다양한 모델이 있으니, 화면 크기(13인치, 15인치 등), RAM 용량, 저장 용량(SSD) 등을 고려하셔야 합니다.\n\n2. **구매처**: 온라인 스토어(예: 삼성 공식 웹사이트, 전자제품 쇼핑몰 등)와 오프라인 매장에서 가격과 혜택(할인, 쿠폰 등)을 비교해보세요.\n\n3. **보증 및 서비스**: 제품 구입 후 보증 기간 및 서비스 정책을 확인하세요.\n\n4. **액세서리**: 필요하신 경우, 가방, 마우스, 외장 하드 드라이브 등 추가 액세서리도 고려해보세요.\n\n혹시 더 필요한 정보나 도움이 필요하시면 언제든지 말씀해 주세요! 구매 후에도 궁금한 점이 있으면 도와드리겠습니다.'

In [44]:
len(store['user_001'].messages)

6

In [45]:
for msg in store['user_001'].messages :
    prefix = 'customer' if msg.type =='human' else 'sales person'
    print(f"[{prefix}] - {msg.content[:30]}")

[customer] - 안녕하세요, 노트북을 사려는데 어떤게 있나요?
[sales person] - 안녕하세요! 노트북을 구매하시려는군요. 용도에 따라 다
[customer] - 가장 인기 있는 제품의 가격이 어떻게 되나요?
[sales person] - 인기 있는 노트북 제품들의 가격은 모델과 스펙에 따라 
[customer] - 좋아요, 2번째 제품으로 구입하겠습니다.
[sales person] - 삼성 노트북 9을 선택해 주셨군요! 정말 좋은 선택입니


In [46]:
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory

In [47]:
window_memory = ConversationBufferWindowMemory(k=2, return_messages = True)
window_memory.save_context({'input':'첫번째, 내 이름은 abc입니다'}, {'output' :'안녕하세요, abc님'})
window_memory.save_context({'input':'두번째, 나는 학생입니다'}, {'output' :'학생이시군요! 멋지십니다'})
window_memory.save_context({'input':'세번째, 나는 파이썬을 좋아합니다'}, {'output' :'파이썬은 정말 재미있죠!'})


/tmp/ipykernel_2868257/790094492.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=2, return_messages = True)


In [48]:
window_memory.load_memory_variables({})

{'history': [HumanMessage(content='두번째, 나는 학생입니다', additional_kwargs={}, response_metadata={}),
  AIMessage(content='학생이시군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='세번째, 나는 파이썬을 좋아합니다', additional_kwargs={}, response_metadata={}),
  AIMessage(content='파이썬은 정말 재미있죠!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [49]:
from langchain_core.messages import trim_messages

messages = []

conversations = [
    ('첫번째, 내 이름은 abc입니다', '안녕하세요, abc님'),
    ('두번째, 나는 학생입니다', '학생이시군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다', '파이썬은 정말 재미있죠!')
]

for user_msg, ai_msg in conversations:
    messages.append(HumanMessage(content=user_msg))
    messages.append(AIMessage(content=ai_msg))
    
messages

[HumanMessage(content='첫번째, 내 이름은 abc입니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='두번째, 나는 학생입니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이시군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬을 좋아합니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬은 정말 재미있죠!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [50]:
trimmer = trim_messages(max_tokens = 4, strategy='last', token_counter=len, start_on='human')
trimmed = trimmer.invoke(messages)

In [51]:
trimmed

[HumanMessage(content='두번째, 나는 학생입니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이시군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬을 좋아합니다', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬은 정말 재미있죠!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [57]:
messages = []
K = 2
def add_turn(user_input, ai_output):
    global messages
    messages.append(HumanMessage(content=user_input))
    messages.append(AIMessage(content=ai_output))
    
    messages = messages[-(K*2):]


conversations = [
    ('첫번째, 내 이름은 abc입니다', '안녕하세요, abc님'),
    ('두번째, 나는 학생입니다', '학생이시군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다', '파이썬은 정말 재미있죠!')
]

add_turn('첫번째, 내 이름은 abc입니다', '안녕하세요, abc님')
print("11111")
print(messages)
add_turn('두번째, 나는 학생입니다', '학생이시군요! 멋지십니다')
print("22222")
print(messages)
add_turn('세번째, 나는 파이썬을 좋아합니다', '파이썬은 정말 재미있죠!')
print("33333")
print(messages)


11111
[HumanMessage(content='첫번째, 내 이름은 abc입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
22222
[HumanMessage(content='첫번째, 내 이름은 abc입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='두번째, 나는 학생입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='학생이시군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
33333
[HumanMessage(content='두번째, 나는 학생입니다', additional_kwargs={}, response_metadata={}), AIMessage(content='학생이시군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='세번째, 나는 파이썬을 좋아합니다', additional_kwargs={}, response_metadata={}), AIMessage(content='파이썬은 정말 재미있죠!', additional_kwargs={}, response_metadata={}, tool

In [58]:
from langchain_classic.memory import ConversationSummaryMemory

In [60]:
summary_llm = ChatOpenAI(model = 'gpt-4o-mini')
summary_memory = ConversationSummaryMemory(
    llm = summary_llm,
    return_messages = False
)

conversations = [
    ('첫번째, 내 이름은 abc입니다', '안녕하세요, abc님'),
    ('두번째, 나는 학생입니다', '학생이시군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다', '파이썬은 정말 재미있죠!')
]

for user_msg, ai_msg in conversations:
    summary_memory.save_context({'input' : user_msg}, {'output' : ai_msg})

/tmp/ipykernel_2868257/2469713396.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(


In [62]:
result = summary_memory.load_memory_variables({})

In [63]:
result['history']

'The human introduces themselves as abc. The AI greets abc. The human then states that they are a student, and the AI praises them for being a student. The human adds that they like Python, to which the AI responds that Python is indeed very fun.'

In [67]:
class SummaryMemory:
    
    def __init__(self, summary_interval = 3):
        self.summary = ''
        self.recent_messages = []
        self.summary_interval = summary_interval
        self.turn_count = 0
        
    def _summarize(self, text):
        response = llm.invoke([
            SystemMessage(content = '주어진 대화 내용을 핵심만 간결하게 요약하세요. 한국어로 작성하세요'),
            HumanMessage(content = f'기존 요약:\n{self.summary}\n\n새 대화:\n{text}')
        ])
        return response.content
    
    def add_exchange(self, user_msg, ai_msg):
        self.recent_messages.append(f'사용자 : {user_msg}')
        self.recent_messages.append(f'AI : {ai_msg}')
        self.turn_count += 1
        
        if self.turn_count % self.summary_interval == 0:
            conversation_text = '\n'.join(self.recent_messages)
            self.summary = self._summarize(conversation_text)
            self.recent_messages = []
            print(f' 요약 완료 턴 {self.turn_count} 에서 요약')
            
    def get_context(self):
        parts = []
        if self.summary :
            parts.append(f'[이전 대화 요약] {self.summary}')
        if self.recent_messages:
            parts.append(f'[최근 대화]\n' + '\n'.join(self.recent_messages))
        return '\n\n'.join(parts)
            

In [68]:
smem = SummaryMemory(summary_interval=2)

In [69]:
conversations = [
    ('첫번째, 내 이름은 abc입니다', '안녕하세요, abc님'),
    ('두번째, 나는 학생입니다', '학생이시군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다', '파이썬은 정말 재미있죠!')
]

for user_msg, ai_msg in conversations:
    smem.add_exchange(user_msg, ai_msg)

 요약 완료 턴 2 에서 요약


In [70]:
smem.get_context()

'[이전 대화 요약] 사용자가 이름과 학생임을 소개하였고, AI는 인사를 하고 학생임에 대해 긍정적인 반응을 보였다.\n\n[최근 대화]\n사용자 : 세번째, 나는 파이썬을 좋아합니다\nAI : 파이썬은 정말 재미있죠!'

In [71]:
class SummaryChatbot:
    def __init__(self, system_prompt = '당신은 도움이 되는 AI 어시스턴트입니다.', summary_interval=3):
        self.system_prompt = system_prompt
        self.memory = SummaryMemory(summary_interval=summary_interval)
    
    def chat(self, user_input):
        context = self.memory.get_context()
        
        messages = [
            SystemMessage(content =self.system_prompt)
        ]
        if context:
            messages.append(SystemMessage(content = f'대화 맥락 : \n{context}'))
        
        messages.append(HumanMessage(content=user_input))
        
        response = llm.invoke(messages)
        ai_response = response.content
        self.memory.add_exchange(user_input, ai_response)
        return ai_response

In [72]:
bot = SummaryChatbot(
    system_prompt = '당신은 IT 커리어 상담사입니다',
    summary_interval = 2
)

In [73]:
questions = [
    "안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다",
    "AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?",
    "현재 Python은 잘하는데 수학과 통계 기초가 부족합니다",
    "온라인 강의와 학교 중 어떤 것이 효과적일까요?"
]

In [74]:
for q in questions:
    print(f'[user] {q}')
    answer = bot.chat(q)
    print(f'[상담사] {answer}')

[user] 안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다
[상담사] 안녕하세요! 백엔드 개발자로 3년 차라니 멋지네요. 어떤 고민이 있으신가요? 구체적으로 이야기해 주시면 도움이 될 수 있도록 최선을 다하겠습니다.
[user] AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?
 요약 완료 턴 2 에서 요약
[상담사] AI/ML 분야로의 전환을 고려하고 계시군요. 흥미로운 도전입니다! 다음은 준비에 도움이 될 몇 가지 단계입니다:

1. **기초 수학 및 통계학 이해**: AI/ML의 근본적인 원리를 이해하려면 선형 대수, 미적분학, 확률 및 통계에 대한 기초 지식이 필요합니다. 이 분야의 알고리즘은 수학적 원리를 바탕으로 하고 있기 때문입니다.

2. **프로그래밍 언어 학습**: Python은 머신러닝에서 가장 널리 사용되는 언어입니다. 다른 언어를 사용하신다면 Python도 익혀두는 것이 좋습니다. R도 통계 분석에 유용합니다.

3. **ML 기본 개념 습득**: 머신러닝의 기본 개념과 알고리즘(예: 회귀, 분류, 클러스터링 등)을 공부하고 간단한 프로젝트를 수행해 보세요. 온라인 강의나 책을 참고할 수 있습니다.

4. **도구 및 프레임워크 익히기**: TensorFlow, PyTorch, Scikit-learn 등 ML을 위한 라이브러리와 툴을 사용해보세요. 해당 도구들을 활용한 프로젝트를 통해 실력을 쌓는 것이 좋습니다.

5. **프로젝트 경험 쌓기**: 개인 프로젝트를 진행하거나 Kaggle의 경진대회에 참여해보세요. 실제 데이터를 다루고 모델을 만들어보는 경험이 중요합니다.

6. **커뮤니티 참여**: ML 관련 커뮤니티에 참여하여 다른 사람들과 교류하고, 질문하거나 의견을 나누세요. GitHub에서 코드 공유 및 협업하는 것도 좋은 방법입니다.

7. **전문 과정 수료**: 대학의 정규 과정이나 온라인 플랫폼(예: Coursera, edX, Udacity)에서 AI/ML 관련 수업을 듣는 것도 좋은 선택입니다.

8